In [34]:
import pandas as pd
import numpy as np

# 1. ОЧИСТКА ФАЙЛА 631502 (Средний медперсонал)

print("Очистка файла: 631502(рус).xlsx...")

df_nurses_raw = pd.read_excel('/Users/laura/Documents/kazakhstan_healthcare/631502(рус).xlsx', sheet_name='Показатель', header=None)
for i in range(len(df_nurses_raw)):
    if 'код КАТО' in str(df_nurses_raw.iloc[i, :].values).lower():
        header_row = i
        break

df_nurses_raw.columns = df_nurses_raw.iloc[header_row]
df_nurses_clean = df_nurses_raw[header_row+1:].reset_index(drop=True)

df_nurses_clean.rename(columns={df_nurses_clean.columns[0]: 'Код_КАТО', 
                                 df_nurses_clean.columns[1]: 'Регион'}, inplace=True)

years = [col for col in df_nurses_clean.columns if str(col).isdigit()]
for col in years:
    df_nurses_clean[col] = pd.to_numeric(df_nurses_clean[col], errors='coerce')

df_nurses_clean = df_nurses_clean[df_nurses_clean['Код_КАТО'].notna()]
df_nurses_clean['Код_КАТО'] = df_nurses_clean['Код_КАТО'].astype(str).str.strip()

print(f"  - Строк: {len(df_nurses_clean)}")
print(f"  - Колонок: {len(df_nurses_clean.columns)}")
print(f"  - Годы: {years[:5]}...{years[-5:]}")


# 2. ОЧИСТКА ФАЙЛА 631501 (Врачи)

print("\nОчистка файла: 631501 (рус).xlsx...")

df_doctors_raw = pd.read_excel('/Users/laura/Documents/kazakhstan_healthcare/631501 (рус).xlsx', sheet_name='Показатель', header=None)

for i in range(len(df_doctors_raw)):
    if 'код КАТО' in str(df_doctors_raw.iloc[i, :].values).lower():
        header_row = i
        break

df_doctors_raw.columns = df_doctors_raw.iloc[header_row]
df_doctors_clean = df_doctors_raw[header_row+1:].reset_index(drop=True)

df_doctors_clean.rename(columns={df_doctors_clean.columns[0]: 'Код_КАТО', 
                                  df_doctors_clean.columns[1]: 'Регион'}, inplace=True)

for col in years:
    if col in df_doctors_clean.columns:
        df_doctors_clean[col] = pd.to_numeric(df_doctors_clean[col], errors='coerce')

df_doctors_clean = df_doctors_clean[~df_doctors_clean['Регион'].astype(str).str.contains('=', na=False)]
df_doctors_clean = df_doctors_clean[df_doctors_clean['Код_КАТО'].notna()]
df_doctors_clean['Код_КАТО'] = df_doctors_clean['Код_КАТО'].astype(str).str.strip()

print(f"  - Строк: {len(df_doctors_clean)}")
print(f"  - Колонок: {len(df_doctors_clean.columns)}")



Очистка файла: 631502(рус).xlsx...
  - Строк: 21
  - Колонок: 25
  - Годы: []...[]

Очистка файла: 631501 (рус).xlsx...
  - Строк: 21
  - Колонок: 25


In [35]:
import pandas as pd
df_doctors_clean.to_excel('врачи_очищенный.xlsx', index=False)
df_pop_clean.to_excel('население_очищенный.xlsx', index=False)

In [13]:
# 3. ОЧИСТКА ФАЙЛА Численность населения
import pandas as pd
df_raw = pd.read_excel('/Users/laura/Documents/kazakhstan_healthcare/611101 Численность населения (по годам).xlsx',
                       sheet_name='2009-2026',
                       header=None)

all_pop_row = df_raw[df_raw[1] == 'Все население'].index[0]
headers = df_raw.iloc[4].values
headers[1] = 'Регион'
kazakhstan_data = df_raw.iloc[all_pop_row + 1].values
men_row = df_raw[df_raw[1] == 'Мужчины'].index[0]
df = df_raw.iloc[all_pop_row + 2:men_row].copy()
df.columns = headers

df = df.dropna(subset=['Код КАТО'])
df = df[df['Код КАТО'].astype(str).str.strip() != '']

regions_dict = {
    '100000000': 'Абай',
    '110000000': 'Акмолинская',
    '150000000': 'Актюбинская',
    '190000000': 'Алматинская',
    '230000000': 'Атырауская',
    '270000000': 'Западно-Казахстанская',
    '310000000': 'Жамбылская',
    '330000000': 'Жетісу',
    '350000000': 'Карагандинская',
    '390000000': 'Костанайская',
    '430000000': 'Кызылординская',
    '470000000': 'Мангистауская',
    '550000000': 'Павлодарская',
    '590000000': 'Северо-Казахстанская',
    '610000000': 'Туркестанская',
    '620000000': 'Ұлытау',
    '630000000': 'Восточно-Казахстанская',
    '710000000': 'г. Астана',
    '750000000': 'г. Алматы',
    '790000000': 'г. Шымкент'
}

df_clean = df[df['Код КАТО'].astype(str).isin(regions_dict.keys())].copy()
df_clean['Регион'] = df_clean['Код КАТО'].astype(str).map(regions_dict)

year_columns = [col for col in df_clean.columns if col not in ['Код КАТО', 'Регион']]
for col in year_columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

rk_dict = {'Код КАТО': 'Республика Казахстан', 'Регион': 'Республика Казахстан'}
for i, col in enumerate(year_columns):
    if i + 2 < len(kazakhstan_data):
        rk_dict[col] = pd.to_numeric(kazakhstan_data[i + 2], errors='coerce')

rk_df = pd.DataFrame([rk_dict])
df_clean = pd.concat([rk_df, df_clean], ignore_index=True)

df_clean = df_clean[['Код КАТО', 'Регион'] + year_columns]

df_clean.to_excel('все_население_регионы.xlsx', index=False)

# Просто показываем всё
print(df_clean)
print(f"\n✅ Сохранено {df_clean.shape[0]} строк, {df_clean.shape[1]} колонок")

                Код КАТО                  Регион      2009.0      2010.0  \
0   Республика Казахстан    Республика Казахстан  15982352.0  16203173.0   
1              100000000                    Абай    653716.0    654744.0   
2              110000000             Акмолинская    737582.0    733882.0   
3              150000000             Актюбинская    756254.0    763055.0   
4              190000000             Алматинская   1100831.0   1125110.0   
5              230000000              Атырауская    509156.0    521019.0   
6              270000000   Западно-Казахстанская    598199.0    603708.0   
7              310000000              Жамбылская   1019623.0   1033304.0   
8              330000000                  Жетісу    619675.0    625526.0   
9              350000000          Карагандинская   1117344.0   1122900.0   
10             390000000            Костанайская    885737.0    882786.0   
11             430000000          Кызылординская    677093.0    688368.0   
12          